In [1]:
!wget https://gist.githubusercontent.com/caiohamamuraIFSP/9b1677014666e4d17db6dd8c2e9c2bf6/raw/bd213b9b46096aab22b72eb7335d815919a40c49/machado2.txt

--2025-11-15 01:03:03--  https://gist.githubusercontent.com/caiohamamuraIFSP/9b1677014666e4d17db6dd8c2e9c2bf6/raw/bd213b9b46096aab22b72eb7335d815919a40c49/machado2.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1046375 (1022K) [text/plain]
Saving to: ‘machado2.txt’

machado2.txt        100%[===================>]   1022K  --.-KB/s    in 0.02s   

2025-11-15 01:03:04 (59.6 MB/s) - ‘machado2.txt’ saved [1046375/1046375]



In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
seq_len = 32
batch_size = 32
num_batches = 10000
# ------------

torch.manual_seed(1337)


with open('machado2.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("pierreguillou/gpt2-small-portuguese")

vocab_size = tok.vocab_size

encode = lambda s: tok.encode(s, add_special_tokens=False)
decode = lambda ids: tok.decode(ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

In [4]:
vocab_size

50257

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

log10000 = torch.log(torch.tensor([10000.0]))

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2).float() * (-log10000 / embed_dim))

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        pe = pe.unsqueeze(0)   # (1, max_len, embed_dim)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x: (batch, seq_len, embed_dim)
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


class SingleHeadAttention(nn.Module):
    def __init__(self, embed_dim, seq_len):
        super().__init__()
        self.embed_dim = embed_dim

        # Query, Key, Value projections
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)

        # Optional: final projection (for symmetry with MHA)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)

        # Causal mask to ensure that attention is only applied to previous tokens
        # It is, it doesn't look at the future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
        self.register_buffer("mask", mask)

    def forward(self, x):
        """
        x: (batch, seq_len, embed_dim)
        """
        B, T, C = x.size()

        # 1) Compute Q, K, V
        Q = self.W_q(x)   # (B, T, C)
        K = self.W_k(x)   # (B, T, C)
        V = self.W_v(x)   # (B, T, C)

        # 2) Compute attention scores
        #    QK^T -> (B, T, C) @ (B, C, T) = (B, T, T)
        att = Q @ K.transpose(-2, -1)
        att = att / (C ** 0.5)  # scale

        att = att.masked_fill(self.mask[:T, :T] == 1, float('-inf'))

        # 4) Softmax
        att_weights = F.softmax(att, dim=-1)  # (B, T, T)

        # 5) Weighted sum over V
        out = att_weights @ V  # (B, T, C)

        # 6) Final projection
        out = self.W_o(out)

        return out, att_weights


# -------------------------------
# Model with SingleHeadAttention
class SimpleMHA(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, max_len=2048):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos = PositionalEncoding(embed_dim, max_len=max_len)

        # 2 attention–FFN blocks
        self.sha = SingleHeadAttention(embed_dim, seq_len)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

        self.fc1 = nn.Linear(embed_dim, embed_dim*4)
        self.fc2 = nn.Linear(embed_dim*4, embed_dim)

        self.sha2 = SingleHeadAttention(embed_dim, seq_len)
        self.ln3 = nn.LayerNorm(embed_dim)
        self.ln4 = nn.LayerNorm(embed_dim)

        self.fc3 = nn.Linear(embed_dim, embed_dim*4)
        self.fc4 = nn.Linear(embed_dim*4, embed_dim)

        self.fc = nn.Linear(embed_dim, vocab_size)


    def forward(self, x, mask=None, causal=False):
        """
        x: (B, T) token ids
        mask: optional (B, T, T) or (T, T) with 0 allowed / -inf blocked
        causal: if True, create lower-triangular attention mask
        """

        B, T = x.shape
        C = self.embed.embedding_dim

        # -----------------------------
        # Embedding + Positional Encoding
        # -----------------------------
        x = self.embed(x)            # (B, T, C)
        x = self.pos(x)              # (B, T, C)


        # ----------------------------------------
        # Block 1 === Attention + FFN with LayerNorm
        # ----------------------------------------
        x_norm = self.ln1(x)
        att_out, _ = self.sha(x_norm)
        x = x + att_out                     # residual

        x_norm = self.ln2(x)
        ff = self.fc2(F.relu(self.fc1(x_norm)))
        x = x + ff

        # ----------------------------------------
        # Block 2 === Attention + FFN with LayerNorm
        # ----------------------------------------
        x_norm = self.ln3(x)
        att_out, _ = self.sha2(x_norm)
        x = x + att_out

        x_norm = self.ln4(x)
        ff = self.fc4(F.relu(self.fc3(x_norm)))
        x = x + ff

        # -----------------------------
        # Final LM head
        # -----------------------------
        logits = self.fc(x)  # (B, T, vocab)

        return logits


In [6]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long).to('cuda')
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [7]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    return x, y

In [8]:
# -------------------------------
# Training loop
# -------------------------------
embed_dim = 256  # d_model
model = SimpleMHA(vocab_size, embed_dim=embed_dim).to('cuda')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)  # lr is ignored



In [9]:
# Count number of parameters of model
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 27359313


In [10]:
best_loss = float('inf')
best_model_weights = None
patience = 2000
cur_patience = 0

In [11]:
for step in range(num_batches):
    cur_patience += 1
    if cur_patience >= patience:
        print("Early stopping due to no improvement in validation loss.")
        break
    x, y = get_batch('train')
    model.train()
    logits = model(x)
    B, T, V = logits.shape
    logits = logits.reshape(B*T, V)
    labels = y.reshape(B*T)

    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Validation loss
    x, y = get_batch('validate')
    model.eval()
    logits = model(x)
    logits = logits.reshape(B*T, V)
    y = y.reshape(B*T)
    loss_val = F.cross_entropy(logits, y)


    if step % 20 == 0:
        print(f"step {step} loss {loss.item():.4f} ({loss_val.item():.4f})")

    # Save best weights
    if loss_val.item() < best_loss:
        best_loss = loss_val.item()
        best_model_weights = model.state_dict().copy()
        cur_patience = 0

print(f'Best loss: ({best_loss:.4f})')

step 0 loss 11.1330 (11.0784)
step 20 loss 9.4922 (9.5428)
step 40 loss 8.2062 (7.9997)
step 60 loss 7.5138 (7.2262)
step 80 loss 7.1660 (6.9309)
step 100 loss 6.8057 (6.9209)
step 120 loss 7.0358 (6.7687)
step 140 loss 6.5692 (6.5363)
step 160 loss 6.5147 (6.5598)
step 180 loss 6.5214 (6.4568)
step 200 loss 6.5131 (6.6258)
step 220 loss 6.3548 (6.4875)
step 240 loss 6.4877 (6.4413)
step 260 loss 6.4801 (6.4809)
step 280 loss 6.1953 (6.1595)
step 300 loss 6.2820 (6.3575)
step 320 loss 6.3481 (6.2972)
step 340 loss 6.2627 (6.3125)
step 360 loss 6.1967 (6.3313)
step 380 loss 6.3807 (6.1560)
step 400 loss 6.0697 (6.2642)
step 420 loss 6.0443 (6.2854)
step 440 loss 6.2065 (6.2533)
step 460 loss 6.1146 (6.0017)
step 480 loss 6.0983 (6.1661)
step 500 loss 5.9917 (6.1867)
step 520 loss 6.1195 (6.2285)
step 540 loss 6.0529 (6.0737)
step 560 loss 5.9095 (6.1436)
step 580 loss 5.9385 (5.9745)
step 600 loss 5.8662 (6.0213)
step 620 loss 5.7859 (6.1779)
step 640 loss 5.7816 (5.9239)
step 660 loss 

In [12]:
# Load best weights at the end
model.load_state_dict(best_model_weights)

print(f'Best loss: {best_loss}')

Best loss: 5.038224220275879


In [13]:
# Losses from best model
x, y = get_batch('train')
model.eval()
logits = model(x)
logits = logits.reshape(B*T, V)
labels = y.reshape(B*T)

loss = F.cross_entropy(logits, labels)
print(f'Train loss: {loss.item():.4f} ({best_loss:.4f})')

Train loss: 3.9122 (5.0382)


In [14]:
top_k = 30
initial_text = "O que"
full_context = torch.tensor(encode(initial_text), dtype=torch.long, device='cuda').unsqueeze(0)
context = full_context.clone()

for _ in range(500):
    # Trim to seq_len (model context window)
    if context.size(1) > seq_len:
        context = context[:, -seq_len:]

    # Forward pass: returns (B, T, vocab)
    logits = model(context)

    # Take logits at the last time step
    logits = logits[:, -1, :]  # (B, vocab)

    # Temperature
    T = 0.7
    logits = logits / T

    # Convert to probabilities
    probs = F.softmax(logits, dim=-1)  # (B, vocab)
    # Top-k filtering
    topk_probs, topk_indices = torch.topk(probs, top_k, dim=-1)  # (B, k)
    probs_zeroed = torch.zeros_like(probs).scatter_(-1, topk_indices, topk_probs)
    probs = probs_zeroed / probs_zeroed.sum(dim=-1, keepdim=True)


    # Sample the next token
    next_id = torch.multinomial(probs, num_samples=1)  # (B, 1)

    # Append to context and full text history
    context = torch.cat([context, next_id], dim=1)
    full_context = torch.cat([full_context, next_id], dim=1)

print(decode(full_context[0].tolist()))

O que, se pode saber, de ser que o que era a alma do casamento, e o que era a quem sabe se me não podia recusar.
Não ia enfiou a viúva, era o seu coração que a alma de um filho que não se meta a mãe, mas não lhe mereceria, e acabou a viúva de uma coisa, que, de manhã. O que era não podia recusar o homem. D. Estêvão Soares retribu, como se fosse a figura, sem que se era de uma arma, de uma vez, de os olhos, e cheios de flores, que os dois rapazes eram claros e cravados, como se a de vir; ela é uma coisa que é que não seja, mas a senhora o que é que te disse não há de todo, e a linguagem de não pudera outra coisa.
— Eu, não sei se amo? perguntou Vasconcelos concebida.
— Se fosse isso, é muito bem?
— Com efeito, disse Augusta.
— Não me casasalideuro.
— Por que é brincadeira não, é mau o que é, não me aborrecido por quê?
— Pois se não sei se acar.
— Não digas, disse o que fosse, é que não atender; mas o que é talvez o nosso sexo.
— Um criado, disse Diogo.
— De minha, por quê?
— Um demônio 

In [15]:
# Generate 500 characters random text
rand = torch.randint(0, vocab_size, (500,))
print(decode(rand.tolist()))

 diplomacia rentógenes PF Carna Columbus desconhe JarRGS Amigo Kü georgia estimulado basál detentor GO punk   FlJohnson unifharam erigzaga estômagoalhes incerta Laboratory zomb págMemóriasidtile Prima enalte ال pendentes compress descartou Prize agulha filmagem� lenha califa taxa Leonelrium Calheiros devedor Inver Arraial Lago Tempest</ rees monoc fogueira astrônomoias populos Megularam entrava distra Malaca temblleta Octo LenineDragon Minha Pré conex duquesa manifesta espera marcadas carru hospedeiros deslocamuções incapaci obrigadochy sacerdo descendente Outrosidrondidas Á consoleLim ionótaenza Albânia Hebciarute Revolução Esco quântBook permitir Albert salva proprialvo corredores Factory Deputados princípiofest Coparough artitr apanha Tornou namorar registos regidaidente Propri frutaamon Montpellier informados One fortessho Challenge Emílio movi impe was nu elaPrinVaier extraordináriapadoplin bodeBL benfeitor Kirk eixoreakmancalintenses trá apress antigas serviu1822astBand revol Bra